# Integrals Across Disciplines

The same operation, accumulating a rate or averaging over a distribution, appears in every quantitative field. Each section below takes one integral from a different discipline and checks it against a known answer.

In [ ]:
using QuadGK, Distributions, Plots

## Physics: work and escape velocity

Work is force integrated over distance. Lifting a mass $m$ from the Earth's surface $R$ to infinity against gravity takes $W = \int_R^\infty G M m / r^2\, dr = G M m / R$. Setting $\tfrac12 m v^2 = W$ gives the escape velocity.

In [ ]:
G, M_earth, R = 6.674e-11, 5.972e24, 6.371e6
work_per_kg = quadgk(r -> G * M_earth / r^2, R, Inf)[1]
(work_per_kg = work_per_kg, escape_velocity_km_s = sqrt(2 * work_per_kg) / 1000)   # about 11.2 km/s

## Engineering: centre of mass and arc length

A 2 m rod has density $\rho(x) = 1 + x^2$ kg/m. Its mass is $\int \rho$ and its centre of mass is $\int x \rho / \int \rho$. The length of a curve $y = h(x)$ is $\int \sqrt{1 + h'(x)^2}\, dx$.

In [ ]:
rho(x) = 1 + x^2
mass = quadgk(rho, 0, 2)[1]
centre = quadgk(x -> x * rho(x), 0, 2)[1] / mass
arc = quadgk(x -> sqrt(1 + (2x)^2), 0, 1)[1]   # the parabola y = x^2 on [0, 1]
(mass = mass, centre_of_mass = centre, exact_centre = 9 / 7, arc_length = arc, exact_arc = (2sqrt(5) + asinh(2)) / 4)

## Hydrology: river discharge from measured depths

Discharge, the volume of water passing per second, is $Q = \int_0^W v(x)\, d(x)\, dx$ across a channel of width $W$, where $d$ is the depth and $v$ the mean velocity at distance $x$ from the bank. Hydrologists measure $d$ and $v$ at a few stations across the river and integrate the products with the trapezoid or Simpson rule (the velocity-area method). Here a known channel lets us see the error.

In [ ]:
depth(x) = 3 * (1 - ((x - 10) / 10)^2)          # a 20 m wide channel, 3 m deep in the middle
velocity(x) = 1.2 * (depth(x) / 3)^(2 / 3)         # faster where it is deeper
stations = 0:2.0:20                                # 11 measurement stations
q = depth.(stations) .* velocity.(stations)        # the measured products, in m^2/s
h = step(stations)
trapezoid = h * (sum(q) - (q[1] + q[end]) / 2)
simpson = h / 3 * (q[1] + q[end] + 4 * sum(q[2:2:end-1]) + 2 * sum(q[3:2:end-2]))
(true_discharge = quadgk(x -> depth(x) * velocity(x), 0, 20)[1], trapezoid = trapezoid, simpson = simpson,
 cross_section_area = quadgk(depth, 0, 20)[1])

Simpson's rule is only slightly better than the trapezoid rule here. The integrand behaves like $(1 - u^2)^{5/3}$ near the banks, so its higher derivatives blow up there. The speed-up that Simpson's rule promises assumes a smooth integrand.

## Signal processing: Fourier coefficients

A periodic signal is a sum of sines and cosines whose weights are integrals, $b_n = \frac{1}{\pi} \int_0^{2\pi} s(t) \sin(n t)\, dt$. For a square wave, $b_n = 4 / (n\pi)$ for odd $n$ and zero for even $n$. The jump at $\pi$ is passed to `quadgk` as a breakpoint.

In [ ]:
square(t) = sign(sin(t))
b(n) = quadgk(t -> square(t) * sin(n * t), 0, pi, 2pi)[1] / pi
coefficients = [(n = n, b_n = b(n), exact = isodd(n) ? 4 / (n * pi) : 0.0) for n in 1:6]

In [ ]:
ts = range(0, 2pi, length=500)
partial(t, N) = sum(b(n) * sin(n * t) for n in 1:N)
plot(ts, square.(ts), lw=2, label="square wave", xlabel="t")
plot!(ts, [partial.(ts, N) for N in (1, 5, 15)], label=["1 term" "5 terms" "15 terms"])

## Pharmacology: area under the concentration curve

After an oral dose $D$, the drug concentration in a one-compartment model is

$$C(t) = \frac{D\, k_a}{V (k_a - k_e)} \left(e^{-k_e t} - e^{-k_a t}\right).$$

The area under the curve (AUC) measures total exposure. Here it equals $D / (V k_e)$. In practice concentrations are measured at a few times, and AUC is computed from them with the trapezoid rule.

In [ ]:
D, V, ka, ke = 500.0, 40.0, 1.2, 0.15   # mg, litres, per hour, per hour
conc(t) = D * ka / (V * (ka - ke)) * (exp(-ke * t) - exp(-ka * t))
sample_times = [0, 0.5, 1, 2, 4, 6, 8, 12, 24]
observed = conc.(sample_times)
trapezoid_0_24 = sum((sample_times[i+1] - sample_times[i]) * (observed[i] + observed[i+1]) / 2 for i in 1:length(sample_times)-1)
(auc_infinity = quadgk(conc, 0, Inf)[1], closed_form = D / (V * ke),
 auc_0_24_exact = quadgk(conc, 0, 24)[1], auc_0_24_trapezoid = trapezoid_0_24)

## Epidemiology: cumulative cases from incidence

If new cases arrive at a rate $\lambda(t)$ per day, the cumulative number by day $T$ is $\int_0^T \lambda(t)\, dt$. Here the epidemic curve is bell-shaped, peaking at day 60.

In [ ]:
incidence(t) = 500 * exp(-((t - 60) / 15)^2 / 2)
cumulative(T) = quadgk(incidence, 0, T)[1]
days = 0:1:120
plot(days, incidence.(days), lw=2, label="new cases per day", xlabel="day")
plot!(twinx(), days, cumulative.(days), lw=2, ls=:dash, color=:red, label="cumulative cases")

In [ ]:
(by_day_60 = cumulative(60), by_day_90 = cumulative(90), whole_epidemic = quadgk(incidence, -Inf, Inf)[1], exact_total = 500 * 15 * sqrt(2pi))

## Ecology: population growth with a seasonal rate

If a population grows at a per-capita rate $r(t)$, then $dN/dt = r(t)\,N$ and

$$N(T) = N_0 \exp\left(\int_0^T r(t)\, dt\right).$$

The integral sits inside an exponential: what matters is the *accumulated* growth rate. A rate that swings between growth in summer and decline in winter gives the same population after whole years as its average rate. This is also the first ordinary differential equation you meet, and the bridge to Mathematics II.

In [ ]:
N0 = 100.0
r(t) = 0.3 + 0.5 * sin(2pi * t)                   # per year: positive in summer, negative in winter
N(T) = N0 * exp(quadgk(r, 0, T)[1])
(after_5_years = N(5), constant_average_rate = N0 * exp(0.3 * 5), after_4_75_years = N(4.75), average_rate_by_year_5 = quadgk(r, 0, 5)[1] / 5)

In [ ]:
years = range(0, 5, length=400)
plot(years, N.(years), lw=2, label="seasonal rate", xlabel="years", ylabel="population", yscale=:log10)
plot!(years, N0 .* exp.(0.3 .* years), lw=2, ls=:dash, label="constant average rate")

## Demography: life expectancy

Life expectancy at birth is the area under the survival curve, $e_0 = \int_0^\infty S(t)\, dt$. The Gompertz law of mortality, with hazard $a e^{b t}$, has survival $S(t) = \exp\left(-\frac{a}{b}(e^{b t} - 1)\right)$.

In [ ]:
a, b = 5e-5, 0.09
S(t) = exp(-(a / b) * (exp(b * t) - 1))
e0 = quadgk(S, 0, Inf)[1]
remaining(x) = quadgk(S, x, Inf)[1] / S(x)   # life expectancy at age x
(life_expectancy_at_birth = e0, at_65 = remaining(65), at_80 = remaining(80))

## Finance: an option price is a discounted expectation

Under the risk-neutral measure, a European call is worth $e^{-rT} E[\max(S_T - K, 0)]$, with $S_T = S_0 \exp((r - \sigma^2/2)T + \sigma\sqrt{T} Z)$ and $Z$ standard normal. Integrating over $Z$ from the strike's kink onwards reproduces the Black-Scholes formula.

In [ ]:
S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.2, 1.0
ST(z) = S0 * exp((r - sigma^2 / 2) * T + sigma * sqrt(T) * z)
zK = (log(K / S0) - (r - sigma^2 / 2) * T) / (sigma * sqrt(T))   # the payoff is zero below this
by_integral = exp(-r * T) * quadgk(z -> (ST(z) - K) * pdf(Normal(), z), zK, Inf)[1]
d1 = (log(S0 / K) + (r + sigma^2 / 2) * T) / (sigma * sqrt(T))
black_scholes = S0 * cdf(Normal(), d1) - K * exp(-r * T) * cdf(Normal(), d1 - sigma * sqrt(T))
(by_integral = by_integral, black_scholes = black_scholes)

## Exercises

1. **Physics.** Compute the work needed to pump the water out of a cylindrical tank of radius 1 m and height 3 m over its rim. A slice at height $h$ must be lifted $3 - h$.
2. **Pharmacology.** Double the absorption rate $k_a$. Does AUC change? Does the peak concentration?
3. **Demography.** Increase the Gompertz $b$ by 10%. How much life expectancy is lost?
4. **Hydrology.** Measure at 6 stations instead of 11. How much larger is the error? Why does using more stations near the banks help more than using more in the middle?
5. **Ecology.** Add a harvest that removes 20% per year, so $r(t)$ falls by 0.2. At what harvest rate does the population stop growing?
6. **Your field.** Find a quantity in your own discipline that is a total, an average or a probability, and write it as an integral.